In [ ]:
import transformers
import datasets
import pandas as pd
import numpy

MODEL_NAME = "lxyuan/distilbert-base-multilingual-cased-sentiments-student"

label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {0: "negative", 1: "neutral", 2: "positive"}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from transformers import pipeline

prodPipeline = pipeline(
    task="text-classification",
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student",
    use_safetensors=True,
)

testInputs = [
        "I keep getting a 500 Internal Server Error every time I click billing.",
        "The application UI is standard, nothing special.",
        "Wow, the loading speed is incredibly fast! Love it.",
    ]

for input in testInputs:
  print(prodPipeline(input))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'positive', 'score': 0.4336186647415161}]
[{'label': 'neutral', 'score': 0.7449544072151184}]
[{'label': 'positive', 'score': 0.9257780313491821}]


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import Dataset

def tokenize():

    df = pd.read_excel("/content/drive/MyDrive/Dataset/BrandPulse_Sentiment_Training_Dataset.xlsx")

    df["label"] = df["label"].map(label2id)
    dataset = Dataset.from_dict(df)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def preprocessFunction(examples):
        return tokenizer(examples["text"], truncation=True, max_length=28)

    tokenizedDataset = dataset.map(preprocessFunction, batched=True)

    return tokenizedDataset

tokenizedDataset = tokenize()

Map:   0%|          | 0/186 [00:00<?, ? examples/s]

In [ ]:
def train():

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,
                                                               num_labels=3,
                                                               id2label=id2label,
                                                               problem_type="single_label_classification")

    trainingArgs = TrainingArguments(
        output_dir="/content/drive/MyDrive/FineTunedWeights/fine_tuned_sentiment",
        learning_rate=0.00002,
        per_device_train_batch_size=4,
        num_train_epochs=4,
        weight_decay=0.01,
        use_cpu=True
    )

    dataCollator = DataCollatorWithPadding(tokenizer)

    trainer = Trainer(
        model =model,
        args = trainingArgs,
        train_dataset = tokenizedDataset,
        data_collator = dataCollator,
    )

    trainer.train()
    trainer.save_model('/content/drive/MyDrive/FineTunedWeights/brandpulse_sentiment_model')
    tokenizer.save_pretrained("/content/drive/MyDrive/FineTunedWeights/brandpulse_sentiment_model")

train()


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
SAVED_MODEL_PATH = "/content/drive/MyDrive/FineTunedWeights/brandpulse_sentiment_model"

testPipeline = pipeline(
    "text-classification",
    model=SAVED_MODEL_PATH,
    tokenizer=SAVED_MODEL_PATH,
)

print(testPipeline)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

TextClassificationPipeline: {'model': 'DistilBertForSequenceClassification', 'dtype': 'float32', 'device': 'cpu', 'input_modalities': 'text'}


In [ ]:
validation_phrases = [
    "I keep getting a 500 Internal Server Error every time I click billing.",
    "The client interface received an ordinary layout configuration shift.",
    "The live processing performance is incredibly efficient and smooth!"
]

print("\n--- Custom Fine-Tuned Model Test Validation ---\n")

for phrase in validation_phrases:
    # Run inference using your loaded weights
    prediction = testPipeline(phrase)[0]

    label = prediction["label"]
    confidence = prediction["score"]

    print(f"Input Text:  '{phrase}'")
    print(f"Prediction:  {label.upper()} (Confidence: {confidence:.2%})")
    print("-" * 50)


--- Custom Fine-Tuned Model Test Validation ---

Input Text:  'I keep getting a 500 Internal Server Error every time I click billing.'
Prediction:  NEGATIVE (Confidence: 98.92%)
--------------------------------------------------
Input Text:  'The client interface received an ordinary layout configuration shift.'
Prediction:  NEUTRAL (Confidence: 59.35%)
--------------------------------------------------
Input Text:  'The live processing performance is incredibly efficient and smooth!'
Prediction:  POSITIVE (Confidence: 98.02%)
--------------------------------------------------
